# 01 — Cinematica dello scattering n–p

Il toy model ricostruisce il neutrone incidente a partire dal protone di rinculo di un singolo
scattering elastico n–p. Questo notebook verifica i fatti cinematici su cui si basano tutti i casi
successivi: la relazione energia–angolo, l'isotropia nel centro di massa e il modello di risoluzione
del detector.

**Etichette dei risultati.** (a) fatto consolidato · (b) stima toy con assunzioni esplicite ·
(c) verificato con Monte Carlo · (d) ipotesi da testare.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid
from scipy.stats import kstest, norm

from riptide_toy.constants import SEED, SIGMA_EP, SIGMA_THETA, EN_MIN, EN_MAX

plt.rcParams["figure.dpi"] = 110
rng = np.random.default_rng(SEED)
from riptide_toy import kinematics

## Energia del protone di rinculo

A masse uguali, un protone che rincula ad angolo $\theta_p$ rispetto alla direzione del neutrone
riceve $E_p = E_n\cos^2\theta_p$ **(a)**. Nel laboratorio $\theta_p \le 90^\circ$: il protone
non può rinculare all'indietro.

In [ ]:
theta = np.linspace(0, np.pi / 2, 200)
fig, ax = plt.subplots(figsize=(5, 3.5))
for En in (1.0, 3.0, 6.0):
    ax.plot(np.degrees(theta), kinematics.proton_energy(En, theta), label=f"$E_n$ = {En} MeV")
ax.set_xlabel(r"$\theta_p$ [gradi]")
ax.set_ylabel(r"$E_p$ [MeV]")
ax.legend()
plt.show()

### Esempio guidato: un protone a 30°

Un neutrone da $E_n = 3$ MeV urta un protone che rincula a $\theta_p = 30^\circ$.

1. $\cos 30^\circ = \sqrt3/2 = 0.866$, quindi $\cos^2 30^\circ = 3/4 = 0.75$;
2. $E_p = E_n\cos^2\theta_p = 3 \times 0.75 = 2.25$ MeV;
3. al neutrone diffuso resta $E_n - E_p = 0.75$ MeV, ed esce a $90^\circ - 30^\circ = 60^\circ$
   (a masse uguali le due particelle escono ad angolo retto).

Al contrario, misurati $E_p$ e $\theta_p$ si ricava $E_n = E_p/\cos^2\theta_p = 2.25/0.75 = 3$ MeV:
è la formula che il Caso A usa evento per evento.

In [ ]:
En_ex, theta_ex = 3.0, np.radians(30.0)
Ep_ex = kinematics.proton_energy(En_ex, theta_ex)
print(f"cos^2(30 gradi) = {np.cos(theta_ex)**2:.4f}   Ep = {Ep_ex:.4f} MeV   (atteso 2.25)")
print(f"inversione: En = Ep / cos^2 = {Ep_ex / np.cos(theta_ex)**2:.4f} MeV")

## Isotropia nel centro di massa

Nel range del toy (0.5–6 MeV) lo scattering n–p è isotropo nel CM: $\cos\theta_{CM}\sim U(-1,1)$
**(b)**. Il protone esce a $\theta_p = (\pi-\theta_{CM})/2$. Da qui, passo per passo **(a)**:

1. $\cos^2\theta_p = \dfrac{1+\cos 2\theta_p}{2} = \dfrac{1+\cos(\pi-\theta_{CM})}{2} = \dfrac{1-\cos\theta_{CM}}{2}$;
2. se $\cos\theta_{CM}$ è uniforme in $[-1,1]$, allora $u = (1-\cos\theta_{CM})/2$ è uniforme in $[0,1]$;
3. quindi $E_p = E_n u \sim U(0, E_n)$: tutte le energie fra 0 e $E_n$ sono ugualmente probabili;
4. momenti della uniforme: $\langle E_p\rangle = E_n/2$ e $\mathrm{Var}(E_p) = E_n^2/12$.
   Con $E_n = 3$ MeV: $\langle E_p\rangle = 1.5$ MeV, $\mathrm{Var}(E_p) = 9/12 = 0.75$ MeV²;
5. distribuzione dell'angolo: $u = \cos^2\theta_p$ è uniforme, quindi
   $p(\theta_p) = |du/d\theta_p| = 2\sin\theta_p\cos\theta_p = \sin 2\theta_p$ su $[0, \pi/2]$;
6. per steradiante si divide per l'elemento d'angolo solido $2\pi\sin\theta_p$:
   $\sin 2\theta_p/(2\pi\sin\theta_p) = \cos\theta_p/\pi$ sull'emisfero anteriore, zero dietro.

Le verifichiamo con $10^5$ eventi a $E_n = 3$ MeV **(c)**.

In [ ]:
En = 3.0
n = 100_000
omega = np.array([0.0, 0.0, 1.0])
Ep, track = kinematics.sample_recoil_events(rng, np.full(n, En), omega)
theta_p = kinematics.recoil_angle_from_direction(track, omega[None, :])[:, 0]

print(f"<Ep>   = {Ep.mean():.4f}   atteso En/2    = {En / 2:.4f}")
print(f"Var Ep = {Ep.var():.4f}   atteso En^2/12 = {En**2 / 12:.4f}")
print(f"max theta_p = {np.degrees(theta_p.max()):.2f} gradi")
print(f"KS cos^2(theta_p) vs U(0,1): p-value = {kstest(np.cos(theta_p)**2, 'uniform').pvalue:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(Ep, bins=60, density=True, alpha=0.7)
axes[0].axhline(1 / En, color="k", ls="--", label="U(0, $E_n$)")
axes[0].set_xlabel("$E_p$ [MeV]"); axes[0].legend()
axes[1].hist(theta_p, bins=60, density=True, alpha=0.7)
axes[1].plot(theta, np.sin(2 * theta), "k--", label=r"$\sin 2\theta_p$")
axes[1].set_xlabel(r"$\theta_p$ [rad]"); axes[1].legend()
plt.tight_layout(); plt.show()

Nota: un generatore che estrae $\theta_p\sim U(0,\pi/2)$ **non** è fisico. Si concentrerebbe
troppo a grandi angoli e non darebbe $E_p$ uniforme. Nel Caso A questa semplificazione non fa
danni, perché lì la verosimiglianza usa lo stesso prior uniforme su $\theta_p$ e il modello resta
coerente. Nei Casi B e C, invece, la direzione della traccia è un'osservabile, e la sua
distribuzione $\cos\theta_p/\pi$ entra nella verosimiglianza.

## Risoluzione del detector

Il toy usa $\sigma_{E_p} = 0.10$ MeV e $\sigma_\theta = 0.08$ rad (valori del caso di riferimento R1). Per
simulare la risoluzione angolare di una traccia 3D, si sposta la direzione vera nel piano tangente
con una gaussiana isotropa di larghezza $\sigma_\theta$ per componente **(b)**. Ci si aspetta:

* la componente dello spostamento lungo un asse è $N(0,\sigma_\theta)$;
* l'angolo totale di deviazione segue una Rayleigh di parametro $\sigma_\theta$.

In [ ]:
track_hat = kinematics.smear_direction(rng, track, SIGMA_THETA)
delta = np.arccos(np.clip(np.sum(track * track_hat, axis=1), -1, 1))
x = np.linspace(0, 0.4, 200)
print(f"rms deviazione = {np.sqrt(np.mean(delta**2)):.4f} rad, atteso sqrt(2)*sigma = {np.sqrt(2) * SIGMA_THETA:.4f}")

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.hist(delta, bins=60, density=True, alpha=0.7, label="simulato")
ax.plot(x, x / SIGMA_THETA**2 * np.exp(-x**2 / (2 * SIGMA_THETA**2)), "k--", label=r"Rayleigh($\sigma_\theta$)")
ax.set_xlabel("deviazione angolare [rad]"); ax.legend()
plt.show()

Un effetto importante della risoluzione angolare: la traccia misurata può avere
$\hat\theta_p > 90^\circ$ rispetto alla direzione vera del neutrone quando il protone esce quasi
perpendicolare. Una verosimiglianza che scartasse come impossibili questi eventi ($-\infty$)
risulterebbe troppo rigida. Il notebook 03 mostra le conseguenze.

In [ ]:
theta_obs = kinematics.recoil_angle_from_direction(track_hat, omega[None, :])[:, 0]
print(f"frazione di tracce misurate oltre 90 gradi: {np.mean(theta_obs > np.pi / 2):.4f}")

### Stima: quante tracce finiscono oltre 90°?

Vicino a $90^\circ$ scriviamo $\delta = \pi/2 - \theta_p$. La densità $\sin 2\theta_p$ diventa
$\sin 2\delta \approx 2\delta$. Una traccia passa oltre $90^\circ$ se la componente polare dello
spostamento, circa $N(0,\sigma_\theta)$, supera $\delta$. Integrando **(b)**:
$$P(\hat\theta_p > 90^\circ) \approx \int_0^\infty 2\delta\;\Phi(-\delta/\sigma_\theta)\,d\delta
= 2\sigma_\theta^2\int_0^\infty x\,\Phi(-x)\,dx = 2\sigma_\theta^2\cdot\frac14 = \frac{\sigma_\theta^2}{2}.$$
Qui $\int_0^\infty x\Phi(-x)dx = 1/4$ si ottiene per parti: vale $\frac12\int_0^\infty x^2\varphi(x)dx = \frac12\cdot\frac12$.
Con $\sigma_\theta = 0.08$ rad: $0.08^2/2 = 0.0032$, cioè circa 3 tracce su mille. Allo stesso modo
la deviazione angolare media della Rayleigh è $\sigma_\theta\sqrt{\pi/2} = 0.100$ rad.

In [ ]:
print(f"stima analitica sigma^2/2 = {SIGMA_THETA**2 / 2:.4f}")
print(f"deviazione media = {delta.mean():.4f} rad, attesa sigma*sqrt(pi/2) = {SIGMA_THETA * np.sqrt(np.pi / 2):.4f}")